In [9]:
import pandas as pd
import requests
import time
from config import coordonate_extinse


In [12]:
print("Tari de procesat:")
print(len(coordonate_extinse), "tari")

Tari de procesat:
74 tari


In [ ]:

# Acum poți folosi variabila direct
tari_de_procesat = list(coordonate_extinse.keys())

PARAMETERS = [
    "T2M", "T2M_MIN", "T2M_MAX", "PRECTOTCORR", 
    "EVPTRNS", "ALLSKY_SFC_SW_DWN", "CDD10", "FROST_DAYS", 
    "GWETROOT" # Umiditatea solului la nivelul rădăcinii (esențială pentru pomi)
]

# Definim sezoanele critice pentru pomicultură
sezoane = {
    "primavara_critica": [3, 4, 5], # Perioada de înflorire (risc de îngheț)
    "vara_crestere":      [6, 7, 8], # Perioada de dezvoltare a fructului (risc de secetă)
}

def fetch_nasa_country(country_name, lat, lon, an_start=2005, an_end=2024):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": ",".join(PARAMETERS),
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": f"{an_start}0101",
        "end": f"{an_end}1231",
        "format": "JSON"
    }

    # Logica de retry și rate limit (păstrată din codul tău)
    for attempt in range(5):
        try:
            r = requests.get(url, params=params, timeout=120)
            if r.status_code == 200: break
            elif r.status_code == 429:
                print(f"  [{country_name}] Rate limit, astept 65s...")
                time.sleep(65)
            else:
                print(f"  [{country_name}] HTTP {r.status_code}")
                return None
        except Exception as e:
            print(f"  [{country_name}] Eroare: {e}")
            return None
    else: return None

    data = r.json()
    daily = data["properties"]["parameter"]
    dates = list(daily["T2M"].keys())
    df = pd.DataFrame({"date": pd.to_datetime(dates, format="%Y%m%d")})
    df["An"] = df["date"].dt.year
    df["luna"] = df["date"].dt.month

    for param in PARAMETERS:
        if param in daily:
            df[param] = list(daily[param].values())
            df[param] = df[param].replace(-999, float("nan"))

    rows = []
    for an in range(an_start, an_end + 1):
        df_an = df[df["An"] == an].copy()
        if df_an.empty: continue
        
        row = {"Area": country_name, "Year": an} # Păstrăm numele coloanelor ca în FAO

        # Agregare Anuală (Exemple cheie)
        row["Temp_Medie_An"] = round(df_an["T2M"].mean(), 2)
        row["Precip_Tot_An"] = round(df_an["PRECTOTCORR"].sum(), 1)
        row["Zile_Inghet_An"] = round(df_an["FROST_DAYS"].sum(), 0)

        # Agregare Sezonieră (Focus pe ce contează pentru mere)
        for sezon, luni in sezoane.items():
            df_s = df_an[df_an["luna"].isin(luni)]
            if not df_s.empty:
                row[f"Temp_Max_{sezon}"] = round(df_s["T2M_MAX"].max(), 2)
                row[f"Precip_{sezon}"]     = round(df_s["PRECTOTCORR"].sum(), 1)
                row[f"Inghet_{sezon}"]     = round(df_s["FROST_DAYS"].sum(), 0)
                row[f"Umid_Sol_{sezon}"]   = round(df_s["GWETROOT"].mean(), 3)

        rows.append(row)

    return pd.DataFrame(rows)

# --- Execuția și Unirea cu Dataset-ul tău ---

results = []
items = list(coordonate_extinse.items())

for i, (country, (lat, lon)) in enumerate(items, 1):
    print(f"[{i}/{len(items)}] Procesez: {country}...")
    df_c = fetch_nasa_country(country, lat, lon)
    if df_c is not None:
        results.append(df_c)
    time.sleep(2) # Poți scădea la 1s dacă API-ul permite

# Concatenăm toate datele meteo
df_clima_final = pd.concat(results, ignore_index=True)

# Încărcăm dataset-ul tău de mere (cel pe care mi l-ai trimis)
df_mere = pd.read_csv("Dataset_Mere_Final.csv")

# UNIREA (Merge): Combinăm datele FAO cu datele Meteo NASA
# Folosim 'inner' pentru a păstra doar rândurile unde avem ambele tipuri de date
df_ml_ready = pd.merge(df_mere, df_clima_final, on=["Area", "Year"], how="inner")

# Salvare
df_ml_ready.to_csv("Dataset_Mere_Clima_Final.csv", index=False)
print("\nDataset gata pentru antrenare!")
print(df_ml_ready.head())

[1/74] Procesez: Afghanistan...


KeyboardInterrupt: 